# Lab 4 - Partea 1: datele si retelele fully connected

In acest notebook:
1. **Datele**: cate clase, cate imagini pe clasa, ce dimensiuni au pozele originale.
2. **Experimentul de baza** (`baseline`): reteaua convolutionala de baza din `lab4_common.DEFAULT_CONFIG`. Este referinta pentru toata echipa.
3. **Augmentarea**: aceeasi retea, cu si fara augmentare.
4. **Rezolutia**: aceeasi retea pe imagini 32 x 32, 64 x 64 si 96 x 96 (compromis viteza / calitate).
5. **Retele fully connected** (0 straturi convolutionale): 1-3 straturi ascunse, diferite numere de neuroni si functii de activare.
6. **De ce o retea fully connected e slaba la imagini.**

Toate experimentele folosesc aceleasi 5 fold-uri si acelasi mod de antrenare (`lab4_common.run_experiment`).
Daca rularea se intrerupe, la rularea urmatoare experimentele (si fold-urile) deja terminate se sar.

In [ ]:
import sys
import json
from pathlib import Path
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), None)
assert ROOT is not None, "Ruleaza notebook-ul din interiorul folderului proiectului (ames-housing-ML)."
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image
from sklearn.metrics import f1_score
import lab4_common

device = lab4_common.get_device()
print("device:", device)

## 1. Datele

### 1.1 Clasele si numarul de imagini

In [ ]:
df = lab4_common.list_images()
counts = df["class"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(counts.index, counts.values, color="#4C72B0")
ax.axhline(len(df) / len(counts), color="gray", ls="--", label="media pe clasa")
for i, v in enumerate(counts.values):
    ax.text(i, v + 10, str(v), ha="center")
ax.set(ylabel="nr. imagini", title=f"Imagini pe clasa (total {len(df)})")
ax.legend()
plt.show()

print(f"Raport clasa cea mai mare / cea mai mica: {counts.max() / counts.min():.2f}")

Sunt 5 clase si 3670 de imagini. Clasele nu sunt perfect egale: **dandelion** are cele mai multe poze (898),
**daisy** cele mai putine (633), adica de aproximativ 1.4 ori mai putine.
Dezechilibrul este mic, deci nu e nevoie de ponderi pe clase sau de oversampling, dar:
- folosim **macro-F1** (media F1 pe clase, fiecare clasa conteaza la fel), nu doar acuratetea;
- fold-urile sunt **stratificate**, ca fiecare fold sa aiba aceleasi proportii de clase.

### 1.2 Dimensiunile pozelor originale

Citim doar dimensiunea fiecarei poze (fara sa incarcam imaginea), ca sa vedem cat de mari si de ce forma sunt.

In [ ]:
sizes = pd.DataFrame(
    [Image.open(lab4_common.DATA_DIR / p).size for p in df["path"]], columns=["latime", "inaltime"]
)
sizes["class"] = df["class"]
sizes["raport"] = sizes["latime"] / sizes["inaltime"]

display(sizes[["latime", "inaltime", "raport"]].describe().round(2))

fig, ax = plt.subplots(figsize=(6, 4))
for cls, g in sizes.groupby("class"):
    ax.scatter(g["latime"], g["inaltime"], s=8, alpha=0.5, label=cls)
ax.set(xlabel="latime (pixeli)", ylabel="inaltime (pixeli)", title="Dimensiunea pozelor originale")
ax.legend(markerscale=2)
plt.show()

Pozele **nu au aceeasi dimensiune**: cea mai frecventa este 320 x 240 (mediana), multe au latura mare de 500 de pixeli,
iar cea mai mare are 1024 de pixeli latime. Si forma difera: aproximativ 82% sunt "landscape" (latimea mai mare decat inaltimea),
13% sunt "portrait" si doar 6% sunt patrate.

O retea are nevoie de intrari de aceeasi marime, asa ca in `lab4_common` fiecare poza este:
1. taiata la un **patrat din centru** (floarea este de obicei in centru, iar asa nu se deformeaza imaginea);
2. **micsorata** la `img_size x img_size` (implicit 64 x 64).

O poza de 320 x 240 are 76.800 de pixeli x 3 canale = peste 230.000 de valori; la 64 x 64 raman 12.288 de valori.
Asta face antrenarea mult mai rapida si permite tinerea tuturor imaginilor in memorie. In sectiunea 4 verificam cat pierdem din calitate.

### 1.3 Cum arata pozele dupa micsorare

In [ ]:
examples = df.groupby("class").sample(2, random_state=0)
small = {size: lab4_common.load_dataset(size)[1] for size in [32, 64, 96]}   # imaginile micsorate (din cache)

fig, axes = plt.subplots(3, 10, figsize=(16, 5.2))
for j, (i, row) in enumerate(examples.iterrows()):
    for r, size in enumerate([32, 64, 96]):
        axes[r, j].imshow(small[size][i].permute(1, 2, 0).numpy())
        axes[r, j].set_xticks([])
        axes[r, j].set_yticks([])
        if r == 0:
            axes[r, j].set_title(row["class"], fontsize=9)
        if j == 0:
            axes[r, j].set_ylabel(f"{size} x {size}")
plt.tight_layout()
plt.show()

La 32 x 32 florile inca se recunosc dupa culoare si forma generala, dar detaliile (petale, centrul florii) se pierd.
La 64 x 64 si 96 x 96 detaliile sunt vizibile. Diferenta dintre 64 si 96 este mica pentru ochi, dar 96 x 96 are de 2.25 ori mai multi pixeli.

## Functii ajutatoare pentru acest notebook

- `run_all(experimente)` ruleaza o lista de experimente (fiecare = numele + modificarile fata de configurarea de baza);
- `show_experiment(name)` deseneaza pentru un experiment: loss si macro-F1 pe epoci (train vs validare, media pe cele 5 fold-uri) si matricea de confuzie;
- `table(names)` arata rezultatele principale ale experimentelor intr-un tabel.

In [ ]:
def run_all(experiments, base=None):
    for name, changes in experiments.items():
        cfg = lab4_common.make_config(**{**(base or {}), **changes, "name": name})
        lab4_common.run_experiment(cfg, verbose=False)


def show_experiment(name):
    fig, axes = plt.subplots(1, 3, figsize=(17, 3.8), gridspec_kw={"width_ratios": [1, 1, 0.9]})
    lab4_common.plot_history(name, ax=axes[:2])
    lab4_common.plot_confusion(name, ax=axes[2])
    plt.show()


COLUMNS = ["name", "val_f1_mean", "val_f1_std", "train_f1_mean", "val_acc_mean", "best_epoch_mean",
           "train_minutes_total", "n_params", "model_mb"]


def table(names):
    log = lab4_common.load_log().set_index("name").loc[names].reset_index()
    return log[COLUMNS].round(3)

## 2. Experimentul de baza (`baseline`)

Reteaua convolutionala de baza: 3 straturi convolutionale (32, 64, 128 filtre, kernel 3 x 3, ReLU, max-pooling 2 x 2),
un strat ascuns de 128 de neuroni, imagini 64 x 64, cu augmentare, Adam cu learning rate 0.001, maxim 30 de epoci, early stopping cu patience 5.

Acesta este punctul de referinta pentru toata echipa: celelalte experimente schimba cate un singur lucru fata de el.
**Il ruleaza doar Partea 1** (colegele il primesc prin git), ca sa nu existe doua foldere `runs/baseline` diferite.

In [ ]:
run_all({"baseline": {}})
table(["baseline"])

In [ ]:
show_experiment("baseline")

## 3. Augmentarea: cu vs fara

Aceeasi retea de baza, antrenata o data cu augmentare (`baseline`) si o data fara (`baseline_noaug`).

In [ ]:
run_all({"baseline_noaug": {"augment": False}})
table(["baseline", "baseline_noaug"])

In [ ]:
show_experiment("baseline_noaug")

## 4. Rezolutia: 32 x 32, 64 x 64, 96 x 96

Aceeasi retea de baza, pe imagini de marimi diferite. Ne intereseaza compromisul dintre calitate (macro-F1) si timp.
Numarul de parametri creste cu rezolutia, pentru ca harta de trasaturi de la ultimul strat convolutional (care intra in stratul fully connected) e mai mare.

In [ ]:
run_all({"res32": {"img_size": 32}, "res96": {"img_size": 96}})
res = table(["res32", "baseline", "res96"])
res.insert(1, "img_size", [32, 64, 96])
res

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].errorbar(res["img_size"], res["val_f1_mean"], yerr=res["val_f1_std"], marker="o", capsize=4)
axes[0].set(xlabel="img_size", ylabel="macro-F1 validare", title="Calitate vs rezolutie", xticks=[32, 64, 96])
axes[1].bar(res["img_size"].astype(str), res["train_minutes_total"], color="#DD8452")
axes[1].set(xlabel="img_size", ylabel="minute (5 fold-uri)", title=f"Timp de antrenare ({device})")
for a in axes:
    a.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
show_experiment("res32")
show_experiment("res96")

## 5. Retele fully connected (0 straturi convolutionale)

Fara straturi convolutionale, imaginea 64 x 64 x 3 este "intinsa" intr-un vector de 12.288 de valori, care intra direct in straturile fully connected.

Pornim de la o retea cu un strat ascuns de 256 de neuroni si ReLU (`fc_256`) si variem pe rand:
- **adancimea si latimea**: 1, 2 sau 3 straturi ascunse, cu 64 - 1024 de neuroni;
- **functia de activare** (pe reteaua cu 2 straturi, 256 si 128): ReLU, LeakyReLU, Tanh, GELU;
- **augmentarea**, si la reteaua fully connected.

Restul setarilor sunt ca la `baseline` (64 x 64, Adam, lr 0.001, early stopping).

In [ ]:
FC = {"conv_layers": []}

FC_SIZES = {
    "fc_64":          {"fc_layers": [64]},
    "fc_256":         {"fc_layers": [256]},
    "fc_1024":        {"fc_layers": [1024]},
    "fc_256_128":     {"fc_layers": [256, 128]},
    "fc_1024_256":    {"fc_layers": [1024, 256]},
    "fc_256_128_64":  {"fc_layers": [256, 128, 64]},
}
FC_ACTIVATIONS = {
    "fc_256_128_leaky": {"fc_layers": [256, 128], "activation": "leaky_relu"},
    "fc_256_128_tanh":  {"fc_layers": [256, 128], "activation": "tanh"},
    "fc_256_128_gelu":  {"fc_layers": [256, 128], "activation": "gelu"},
}
FC_OTHER = {
    "fc_256_noaug": {"fc_layers": [256], "augment": False},
}

run_all(FC_SIZES, base=FC)
run_all(FC_ACTIVATIONS, base=FC)
run_all(FC_OTHER, base=FC)

### 5.1 Adancimea si numarul de neuroni

In [ ]:
table(list(FC_SIZES))

In [ ]:
for name in FC_SIZES:
    show_experiment(name)

### 5.2 Functia de activare

In [ ]:
act = table(["fc_256_128", *FC_ACTIVATIONS])
act.insert(1, "activation", ["relu", "leaky_relu", "tanh", "gelu"])
act

In [ ]:
for name in FC_ACTIVATIONS:
    show_experiment(name)

### 5.3 Augmentarea la reteaua fully connected

In [ ]:
table(["fc_256", "fc_256_noaug"])

In [ ]:
show_experiment("fc_256_noaug")

## 6. De ce o retea fully connected e slaba la imagini

### 6.1 Calitate vs numar de parametri

Fiecare punct este un experiment din acest notebook. Axa x (numarul de parametri) este logaritmica.

In [ ]:
ALL = ["baseline", "baseline_noaug", "res32", "res96", *FC_SIZES, *FC_ACTIVATIONS, *FC_OTHER]
log = lab4_common.load_log().set_index("name").loc[ALL].reset_index()
is_fc = log["conv_layers"] == "[]"

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(log.loc[is_fc, "n_params"], log.loc[is_fc, "val_f1_mean"], s=50, label="fully connected (0 conv)")
ax.scatter(log.loc[~is_fc, "n_params"], log.loc[~is_fc, "val_f1_mean"], s=50, marker="s", label="convolutional (3 conv)")
for _, r in log.iterrows():
    ax.annotate(r["name"], (r["n_params"], r["val_f1_mean"]), fontsize=7, xytext=(4, 3), textcoords="offset points")
ax.set_xscale("log")
ax.set(xlabel="numar de parametri (scala log)", ylabel="macro-F1 validare", title="Calitate vs marimea retelei")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

log[["name", "val_f1_mean", "n_params", "model_mb", "train_minutes_total"]].round(3)

### 6.2 Ce "vede" primul strat al unei retele fully connected

Fiecare neuron din primul strat ascuns are cate o greutate pentru fiecare pixel si canal (64 x 64 x 3 = 12.288 de greutati).
Putem desena aceste greutati ca pe o imagine: arata "sablonul" pe care il cauta neuronul in toata poza.
Folosim modelul `fc_256` salvat din fold-ul 0.

In [ ]:
def load_saved_model(name):
    cfg = json.loads((lab4_common.OUT_DIR / "runs" / name / "config.json").read_text())
    model = lab4_common.build_model(cfg)
    model.load_state_dict(torch.load(lab4_common.OUT_DIR / "models" / f"{name}.pt", map_location="cpu"))
    return cfg, model.eval()


cfg_fc, model_fc = load_saved_model("fc_256")
W = model_fc.classifier[0].weight.detach().reshape(-1, 3, 64, 64)   # (neuroni, 3, 64, 64)

fig, axes = plt.subplots(2, 8, figsize=(16, 4.4))
for ax, w in zip(axes.ravel(), W[:16]):
    w = (w - w.min()) / (w.max() - w.min())
    ax.imshow(w.permute(1, 2, 0).numpy())
    ax.axis("off")
fig.suptitle("Greutatile primilor 16 neuroni din primul strat (fc_256)")
plt.show()

### 6.3 Ce se intampla daca mutam imaginea cu cativa pixeli

Luam imaginile de validare din fold-ul 0 si le mutam la dreapta cu 0 - 16 pixeli (ce iese din dreapta intra in stanga).
Floarea ramane aceeasi, doar pozitia ei se schimba. Comparam macro-F1 pentru cea mai buna retea fully connected si pentru `baseline`.

In [ ]:
fc_names = [*FC_SIZES, *FC_ACTIVATIONS, *FC_OTHER]
best_fc = log[log["name"].isin(fc_names)].sort_values("val_f1_mean").iloc[-1]["name"]
print("cea mai buna retea fully connected:", best_fc)

df64, X64, y64 = lab4_common.load_dataset(64)
fold = lab4_common.get_folds(df64)
val = np.where(fold == 0)[0]
shifts = [0, 2, 4, 8, 12, 16]

rows = []
for name in [best_fc, "baseline"]:
    _, model = load_saved_model(name)
    for s in shifts:
        Xs = torch.roll(X64[val], shifts=s, dims=3)
        pred = lab4_common.predict(model, Xs, "cpu").argmax(1)
        rows.append({"model": name, "shift": s, "macro_f1": f1_score(y64[val].numpy(), pred, average="macro")})
shift_res = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(6.5, 3.8))
for name, g in shift_res.groupby("model", sort=False):
    ax.plot(g["shift"], g["macro_f1"], marker="o", label=name)
ax.set(xlabel="mutare la dreapta (pixeli)", ylabel="macro-F1 (fold 0)", title="Sensibilitatea la pozitia florii")
ax.grid(alpha=0.3)
ax.legend()
plt.show()
shift_res.pivot(index="shift", columns="model", values="macro_f1").round(3)

## 7. Rezumatul experimentelor din Partea 1

In [ ]:
table(ALL).sort_values("val_f1_mean", ascending=False, ignore_index=True)